In [1]:
from google.colab import drive
import rasterio
from PIL import Image
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil
import random

In [2]:
# Connect google drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
# Tiles path
tif_dir = Path("/content/gdrive/MyDrive/ParkingSpaceDetection/data/tiles")
png_dir = Path("images")
png_dir.mkdir(parents=True, exist_ok=True)

# Load tiles and convert the tif to png images (necessary for yolo)
for tif_path in tif_dir.glob("*.tif"):
    with rasterio.open(tif_path) as src:
        img = src.read([1, 2, 3])
        img = img.transpose(1, 2, 0)
        out_path = png_dir / (tif_path.stem + ".png")
        Image.fromarray(img).save(out_path)

In [4]:
# Width and height of image
IMG_W = 1024
IMG_H = 1024

# Assign class
class_map = {"parking_space": 0, "1": 0}

# Labels path
xml_dir = Path("/content/gdrive/MyDrive/ParkingSpaceDetection/data/labels")
yolo_dir = Path("labels")
yolo_dir.mkdir(exist_ok=True)

# Load labels (xml files)
for xml_file in xml_dir.glob("*.xml"):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    yolo_lines = []

    # Extract the bounding box values
    for obj in root.findall("object"):
        cls_name = obj.find("name").text

        if cls_name in class_map:
            cls_id = class_map[cls_name]

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            w = xmax - xmin
            h = ymax - ymin
            x_center = xmin + w / 2
            y_center = ymin + h / 2

            yolo_line = f"{cls_id} {x_center/IMG_W:.6f} {y_center/IMG_H:.6f} {w/IMG_W:.6f} {h/IMG_H:.6f}"
            yolo_lines.append(yolo_line)
        else:
            print(f"Warning: Class name '{cls_name}' not found in class_map for file {xml_file.name}.")

    # Convert to text files (necessary for yolo)
    txt_path = yolo_dir / (xml_file.stem + ".txt")
    txt_path.write_text("\n".join(yolo_lines))

In [5]:
# Paths
img_dir = Path("images")
lbl_dir = Path("labels")

train_img_dir = Path("dataset/images/train")
val_img_dir   = Path("dataset/images/val")
train_lbl_dir = Path("dataset/labels/train")
val_lbl_dir   = Path("dataset/labels/val")

# Create folders
for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    d.mkdir(parents=True, exist_ok=True)

# List images
images = list(img_dir.glob("*.png"))

# Split data in training (80%) and validation (20%)
split_idx = int(0.8 * len(images))
train_imgs = images[:split_idx]
val_imgs   = images[split_idx:]

# Function for moving files to directories
def move_pairs(img_list, target_img_dir, target_lbl_dir):
    for img_path in img_list:
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        shutil.copy(img_path, target_img_dir / img_path.name)
        shutil.copy(lbl_path, target_lbl_dir / lbl_path.name)

# Move files
move_pairs(train_imgs, train_img_dir, train_lbl_dir)
move_pairs(val_imgs,   val_img_dir,   val_lbl_dir)


In [6]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 70.2 MB/s eta 0:00:00


In [7]:
from ultralytics import YOLO

# Load pre-trained weights for feature extraction
model = YOLO("yolov9c.pt")

# Load yaml file from drive
yaml = "/content/gdrive/MyDrive/ParkingSpaceDetection/data.yaml"

# Train model
results = model.train(
    data=yaml,
    epochs=100,
    imgsz=1024,
    batch=16,
    device=0,
    single_cls=True,
    rect=True,
    patience=20
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/MyDrive/ParkingSpaceDetection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015,

In [8]:
# Validate model
results = model.val(data=yaml)

Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv9c summary (fused): 156 layers, 25,320,019 parameters, 0 gradients, 102.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4315.9±1149.2 MB/s, size: 1311.0 KB)
val: Scanning /content/dataset/labels/val.cache... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25 7.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.2s/it 4.5s
                   all         25        275      0.792      0.785      0.826      0.469
Speed: 20.7ms preprocess, 97.6ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val
